# Import Pandas and the Train/Test Dataset

In [ ]:
import pandas as pd
train_data=pd.read_csv('./data/train.csv')
test_data = pd.read_csv('./data/test.csv')


# Prepare the data and Feature Engineering

In [ ]:
def prepare_features(df):
    
    df = df.copy()
    df = df.sort_values(['Driver', 'Race', 'LapNumber']) 
    
    
    df['Degradation_Delta'] = df.groupby(['Driver', 'Race'])['Cumulative_Degradation'].diff()
    df['Laptime_Rolling'] = df.groupby('Driver')['LapTime (s)'].transform(lambda x: x.rolling(3).mean())
    
    df = df.fillna(0) 
    return df


train_data = prepare_features(train_data)

cat_cols = ['Driver', 'Compound', 'Race']
for col in cat_cols:
    train_data[col] = train_data[col].astype('category')

# Split the Data

In [5]:
from sklearn.model_selection import train_test_split

X=train_data.drop(['PitNextLap','id'],axis=1)
y=train_data['PitNextLap']

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)


# LightGBM with Optuna 


In [ ]:
import optuna
import joblib
import lightgbm as lgb
from sklearn.model_selection import cross_val_score


def objective(trial):
    params = {
        "n_estimators": 500, 
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "num_leaves": trial.suggest_int("num_leaves", 20, 80), # Πιο μαζεμένο
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
        "subsample": 0.8, 
        "colsample_bytree": 0.8, 
        "class_weight": "balanced",
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }

    model = lgb.LGBMClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=3, scoring="roc_auc", n_jobs=-1)
    return scores.mean()


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))


study.optimize(objective, n_trials=15, show_progress_bar=True)

print(f"--- Best CV AUC-ROC: {study.best_value:.5f} ---")

best_lgb = lgb.LGBMClassifier(**study.best_params, random_state=42, n_jobs=-1, verbose=-1)
best_lgb.fit(X_train, y_train)

y_pred_proba = best_lgb.predict_proba(X_test)[:, 1]



  0%|          | 0/15 [00:00<?, ?it/s]

Best trial: 14. Best value: 0.939699: 100%|██████████| 15/15 [06:37<00:00, 26.50s/it]


--- Best CV AUC-ROC: 0.93970 ---


# Create the submission file


In [ ]:

test_data = pd.read_csv('./data/test.csv')

test_proc = test_data.copy()

test_proc['Degradation_Delta'] = test_proc.groupby(['Driver', 'Race'])['Cumulative_Degradation'].diff().fillna(0)
test_proc['Laptime_Rolling'] = test_proc.groupby(['Driver', 'Race'])['LapTime (s)'].transform(lambda x: x.rolling(3).mean()).fillna(0)

for col in ['Driver', 'Compound', 'Race']:
    test_proc[col] = test_proc[col].astype('category')


X_test_final = test_proc[X_train.columns] 


real_proba = best_lgb.predict_proba(X_test_final)[:, 1]

submission = pd.DataFrame({
    "id": test_data["id"],
    "PitNextLap": real_proba
})

submission.to_csv("submission.csv", index=False)